# HalfKP NNUE pipeline
The reusable implementation lives in Python files. These cells only configure and call it. For full parallel data loading on Windows, prefer the equivalent `python -m evaluation.pipeline ...` commands documented in `evaluation/README.md`.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd().parent if Path.cwd().name == 'evaluation' else Path.cwd()
sys.path.insert(0, str(repo))

from evaluation.baselines import evaluate_baselines, format_metrics
from evaluation.pipeline import train_experiment
from evaluation.training import benchmark_accumulator, evaluate, inspect_data, recommended_workers, train

print(inspect_data())
print('Recommended workers:', recommended_workers())

In [ ]:
architectures = [
    '256x32x1',       # compact
    '256x32x32x1',    # classic HalfKP NNUE
    '512x32x1',       # current baseline
    '1024x32x32x1',   # Stockfish-sized main layers
    '1024x256x32x1',  # larger experiment
]

runs = []
for features in ('piece768', 'halfkp32'):
    for architecture in architectures:
        print(f'\n{features} | {architecture}')
        runs.append(train(
            representation=features,
            architecture=architecture,
            epochs=10, batch_size=8192,
            workers=0, device='cuda', test=False,
        ))

In [ ]:
baselines = evaluate_baselines(split='validation')
for name, metrics in baselines.items(): print(format_metrics(name, metrics))

In [ ]:
smoke = train_experiment(objective='wdl', architecture='512x32x1', epochs=1, workers=0, max_train_batches=20, max_validation_batches=5, test=False)

In [ ]:
for run in runs:
    name = f'{run.representation} {run.architecture}'
    metrics = evaluate(run, split='validation', batch_size=8192, workers=0)
    print(format_metrics(name, metrics))
    if run.representation == 'halfkp32':
        print(benchmark_accumulator(run, iterations=1000))

In [ ]:
import importlib
from evaluation import training as nnue
nnue = importlib.reload(nnue)

for run in runs:
    name = f'{run.representation} {run.architecture}'
    full = nnue.benchmark_inference(run.model_path)
    print(f'{name:<28} full={full.microseconds_per_position:7.1f} us  ({full.positions_per_second:,.0f}/s)')
    if run.representation == 'halfkp32':
        game = nnue.benchmark_game(run.model_path)
        print(f'{"":28} game={game.total_microseconds:7.1f} us  '
              f'update={game.update_microseconds:6.1f} us  '
              f'head={game.inference_microseconds:6.1f} us  '
              f'({game.positions_per_second:,.0f}/s)')

In [ ]:
from evaluation.plotting import plot_tradeoffs

plot_tradeoffs(runs)

## Focused HalfKP width sweep
Keep the head small and test whether wider feature transformers continue to improve accuracy.

In [ ]:
halfkp_architectures = [
    '384x32x1',
    '768x32x1',
    '1024x16x1',
    '1024x32x1',
    '1024x64x1',
    '1536x32x1',
]

new_halfkp_runs = []
for architecture in halfkp_architectures:
    print(f'\nHalfKP | {architecture}')
    new_halfkp_runs.append(train(
        representation='halfkp32',
        architecture=architecture,
        epochs=10, batch_size=8192,
        workers=0, device='cuda', test=False,
    ))

In [ ]:
old_halfkp_runs = [run for run in runs if run.representation == 'halfkp32']
plot_tradeoffs(old_halfkp_runs + new_halfkp_runs)

## Continue the two finalists
Fine-tune the saved best checkpoints at a lower learning rate for 50 more epochs.

In [ ]:
import importlib
from evaluation import training as nnue
nnue = importlib.reload(nnue)

finalists = [
    run for run in new_halfkp_runs
    if run.architecture in {'768x32x1', '1024x32x1'}
]

long_halfkp_runs = []
for run in finalists:
    print(f'\nContinuing {run.architecture}')
    long_halfkp_runs.append(nnue.train(
        representation='halfkp32',
        architecture=run.architecture,
        resume_from=run.model_path,
        epochs=50, learning_rate=3e-4, patience=None,
        batch_size=8192, workers=0, device='cuda',
        test=False,
    ))

In [ ]:
import importlib
from evaluation import plotting
plotting = importlib.reload(plotting)

model_dir = repo / 'evaluation' / 'models'
architectures = ('768x32x1', '1024x32x1')
history_files = [
    max(model_dir.glob(f'halfkp_hm_{architecture}_wdl_*.json'), key=lambda path: path.stat().st_mtime)
    for architecture in architectures
]

plotting.plot_training_curves(history_files)